# ArrayRecovery

[Codility task](https://app.codility.com/programmers/trainings/3/array_recovery/) - Exercise 3, 2017 Contest

An unknown array `A` (values in `[1..M]`) produced `B`, where `B[J]` is the last value **before** `J` that is smaller than `A[J]` (or 0). Given `B` and `M`, `solution(B, M)` returns the number of arrays `A` that produce `B`, modulo `10^9 + 7`.

| M | B | result |
|---|---|--------|
| 4 | `[0, 2, 2]` | 3 |
| 10 | `[0, 3, 5, 6]` | 4 |
| 10^5 | `[0, 0]` | 49965 (5,000,050,000 mod 10^9+7) |

Assumptions: `N` up to 100,000, `M` up to 10^9, and at least one valid `A` exists.

## Idea

"Previous smaller value" is the classic **monotonic stack**: keep strictly increasing values, pop while `top >= A[J]`, then `B[J]` is the top (0 for an empty stack, modelled by a sentinel `0` at the bottom), then push `A[J]`.

Look at the stack after each step:

- every element **except the top** is already revealed by `B` (the element pushed on top of it has `B` equal to it);
- the top is the only unknown.

Group consecutive equal values of `B` into **runs**. A run `B = c` repeated `L` times means `A` contains a non-increasing sequence `a_1 >= a_2 >= ... >= a_L`, all in `(c, hi]`, where `hi` is the stack element just above `c` (or `M` if `c` is the top). Then the next value `v` of `B` tells what happened:

| next `B` | meaning | ways for the run |
|----------|---------|------------------|
| end of array | nothing else | non-increasing sequences of length `L` in `(c, hi]`: `C(hi - c + L - 1, L)` |
| `v > c` | the last element of the run was **exactly** `v` and stays on the stack (`hi` becomes `M`) | `a_L = v`: `C(hi - v + L - 1, L - 1)` |
| `v < c` | the run's elements were popped; `v` is deeper in the stack, and the new `hi` is the element just above `v` | `C(hi - c + L - 1, L)` |

The total is the product over runs. The binomials have a huge top (up to about 10^9 + 10^5, which can exceed the modulus) but a small bottom `k <= N < p`, so `C(n, k) = n (n-1) ... (n-k+1) / k!` evaluated modulo `p` (with the modular inverse of `k!`) is valid. (When the top reaches `p` the product contains a factor `p`, so that run contributes `0`.)

Time O(N), plus one modular inverse per run.

In [ ]:
MOD = 10**9 + 7


def solution(B, M):
    n = len(B)

    # factorials up to N and their inverses (p is prime, k! is invertible for k < p)
    fact = [1] * (n + 1)
    for i in range(1, n + 1):
        fact[i] = fact[i - 1] * i % MOD
    inv_fact = [1] * (n + 1)
    inv_fact[n] = pow(fact[n], MOD - 2, MOD)
    for i in range(n, 0, -1):
        inv_fact[i - 1] = inv_fact[i] * i % MOD

    def comb(top, k):
        """C(top, k) mod p, with top possibly larger than p and k <= N."""
        if k < 0 or top < k:
            return 0
        result = 1
        for i in range(k):
            result = result * ((top - i) % MOD) % MOD
        return result * inv_fact[k] % MOD

    stack = [0]            # known values, strictly increasing; 0 is a sentinel
    hi = M                 # upper bound for the first element of the current run
    answer = 1
    i = 0
    while True:
        c = stack[-1]                              # B value shared by the current run
        j = i
        while j < n and B[j] == c:
            j += 1
        length = j - i
        if length == 0:
            raise ValueError("B is not a valid array")
        if j == n:                                  # last run
            return answer * comb(hi - c + length - 1, length) % MOD
        v = B[j]
        if v > c:                                   # the run's last element is exactly v
            answer = answer * comb(hi - v + length - 1, length - 1) % MOD
            stack.append(v)
            hi = M
        else:                                       # the run was popped; go back to an older stack value
            answer = answer * comb(hi - c + length - 1, length) % MOD
            while stack[-1] != v:
                hi = stack.pop()
                if not stack:
                    raise ValueError("B is not a valid array")
        i = j

## Official examples

In [ ]:
assert solution([0, 2, 2], 4) == 3
assert solution([0, 3, 5, 6], 10) == 4
assert solution([0, 0], 10**5) == 49965
print("examples OK")

## Edge cases

In [ ]:
assert solution([0], 1) == 1                          # A = [1]
assert solution([0], 10**9) == 10**9 % MOD            # any value of the range
assert solution([0, 0, 0], 1) == 1                    # A = [1, 1, 1]
assert solution([0, 1], 2) == 1                       # A = [1, 2]
assert solution([0, 999_999_999], 10**9) == 1         # A = [999999999, 10**9]
assert solution([0, 0], 1) == 1
assert solution([0, 1, 0], 3) == 2                    # A = [1, 2, 1] or [1, 3, 1]: A[0] is pinned to 1, A[2] must be 1
print("edge cases OK")

## Exhaustive brute force

For small `(N, M)` the reference builds `B` from **every** array `A` straight from the definition (looking backwards for the last smaller value) and counts how many `A` give each `B`. The solution must agree for every `B` that can occur.

In [ ]:
import itertools
from collections import Counter


def previous_smaller(A):
    B = []
    for j in range(len(A)):
        value = 0
        for k in range(j - 1, -1, -1):
            if A[k] < A[j]:
                value = A[k]
                break
        B.append(value)
    return tuple(B)


total_B = 0
for N, M in [(1, 5), (2, 6), (3, 6), (4, 5), (5, 4), (6, 4), (7, 3), (8, 3), (9, 3), (10, 2)]:
    counts = Counter(previous_smaller(A) for A in itertools.product(range(1, M + 1), repeat=N))
    for B, expected in counts.items():
        assert solution(list(B), M) == expected % MOD, (B, M, expected)
    total_B += len(counts)
print(f"matches the exhaustive count for {total_B} different arrays B")

## Second reference: dynamic programming over stacks

For larger `N` a different independent method is used. It simulates the process literally: the state is the **whole stack** (a tuple of strictly increasing values) and it counts how many choices of `A[J]` keep producing the right `B[J]`. With a small `M` there are few stacks, so it handles `N` up to 30.

In [ ]:
import random
from collections import defaultdict


def count_by_stack_dp(B, M):
    states = {(): 1}
    for b in B:
        new = defaultdict(int)
        for stack, ways in states.items():
            for a in range(1, M + 1):
                s = list(stack)
                while s and s[-1] >= a:
                    s.pop()
                if (s[-1] if s else 0) == b:
                    new[tuple(s) + (a,)] += ways
        states = new
    return sum(states.values()) % MOD


rng = random.Random(3)
for _ in range(300):
    N = rng.randint(1, 30)
    M = rng.randint(1, 9)
    A = [rng.randint(1, M) for _ in range(N)]
    B = list(previous_smaller(A))
    assert solution(B, M) == count_by_stack_dp(B, M), (A, M)
print("matches the stack DP on 300 random arrays (N up to 30, M up to 9)")

## The modulus: binomials whose top exceeds `p`

With `B = [0] * k` there is a single run and the answer is `C(M + k - 1, k)`. For `M = 10^9` and `k >= 8` the top is larger than `10^9 + 7`. Python's exact `math.comb` gives the reference.

Curious fact: since `top <= M + k - 1 <= p - 8 + k`, whenever `top >= p` we get `top mod p < k`, so the falling factorial contains a multiple of `p` and the binomial is `0` modulo `p` (consistent with Lucas' theorem). The formula handles that case with no special code, and the checks below include it.

In [ ]:
import math

for k in (1, 7, 8, 9, 50, 2_000, 20_000):
    assert solution([0] * k, 10**9) == math.comb(10**9 + k - 1, k) % MOD, k
for k in (1, 5, 300):
    for M in (1, 2, 10**9 - 3, 10**9):
        assert solution([0] * k, M) == math.comb(M + k - 1, k) % MOD, (k, M)
assert solution([0] * 8, 10**9) == 0                  # top = 10^9 + 7 = p exactly
assert solution([0] * 7, 10**9) != 0                  # top = 10^9 + 6 < p
print("single-run answers match math.comb, including tops larger than 10^9 + 7")

## Performance

In [ ]:
import time

N = 100_000


def timed(name, B, M):
    start = time.perf_counter()
    result = solution(B, M)
    print(f"{name:28s} -> {result:>10} in {time.perf_counter() - start:.3f}s")


def previous_smaller_fast(A):
    stack, B = [], []
    for a in A:
        while stack and stack[-1] >= a:
            stack.pop()
        B.append(stack[-1] if stack else 0)
        stack.append(a)
    return B


assert previous_smaller_fast([2, 5, 3, 7, 9, 6]) == [0, 2, 2, 3, 7, 3]

timed("all zeros (one huge run)", [0] * N, 10**9)

# strictly increasing A: every element is pinned by the next one
A = list(range(1, N + 1))
assert previous_smaller_fast(A) == [0] + A[:-1]
timed("strictly increasing", [0] + A[:-1], 10**9)

# random A with a large M and with smaller ones (many runs, many pops)
rng = random.Random(11)
for M in (10**9, 1000, 10):
    A = [rng.randint(1, M) for _ in range(N)]
    timed(f"random A, M = {M}", previous_smaller_fast(A), M)

## Notes

- Understanding *what is known* is the key: in the stack only the top is free; everything else was fixed by `B`.
- Counting non-increasing sequences of length `L` in a range of `r` values is "stars and bars" / multisets: `C(r + L - 1, L)`.
- Never reduce `top` modulo `p` before checking `top >= k`, and remember that the falling-factorial formula stays valid modulo `p` because `k < p`.
- Strategy to trust a hard solution: exhaustive enumeration for tiny inputs, a second independent algorithm for medium inputs, and `math.comb` for the big-number part.